In [1]:
import sys
from pathlib import Path
import numpy as np
import pickle

PROJECT_ROOT = Path("..").resolve()   # because notebook is inside rfa_python/notebooks
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import rfa_model.geometry as geom

importlib.reload(geom)

from rfa_model.geometry import *
from rfa_model.plotting import *
from rfa_model.io import load_field_npz
from rfa_model.fields import *
from rfa_model.trajectories import *

In [2]:
stl_dir = Path("../rfa stl")

meshes, sample_parts = load_and_align_sample_assembly(
    stl_dir,
    alpha_deg=0.0,
)

frame_meshes, frame_parts, frame_info = load_and_align_grid_frames(
    stl_dir,
    verbose=True,
)

sample_y_bounds, sample_z_bounds = sample_bounds(meshes)

Grid frame alignment
  fitted center: [-0.00113118 -0.00140144 -0.00109564]
  fitted radius: 0.05887084242766213
  axis before alignment: [ 9.99990913e-01  4.23261006e-03 -5.08542373e-04]
  R_g1: 0.04519042252564098
  R_g2: 0.05792646490184952
  R_g3: 0.07107616160072151


In [3]:
field = load_field_npz("../results/fields/rfa_field_sample_0_g2g3_0_collector50_0p5mm.npz")

print(field.keys())
print("V shape:", field["V"].shape)
print("h:", field["h"])
print("voltages:", field["voltages"])

dict_keys(['x', 'y', 'z', 'h', 'V', 'Ex', 'Ey', 'Ez', 'R_g1', 'R_g2', 'R_g3', 'R_col', 'fixed', 'update_region', 'Vfix', 'owner', 'voltages', 'Vs', 'Vr', 'Vg1', 'Vg2', 'Vg3', 'Vc', 'Vdt'])
V shape: (333, 333, 333)
h: 0.0005
voltages: {'Vs': 0.0, 'Vr': 0.0, 'Vg1': 0.0, 'Vg2': 0.0, 'Vg3': 0.0, 'Vc': 50.0, 'Vdt': 0.0}


In [4]:
Ex_interp, Ey_interp, Ez_interp = build_field_interpolators(field)
Phi_interp = build_potential_interpolator(field)

p_test = np.array([0.001, 0.0, 0.0])

print("E =", E_at_point(p_test, Ex_interp, Ey_interp, Ez_interp))
print("Phi =", potential_at_point(p_test, Phi_interp))
print("speed 500 eV =", speed_from_energy_eV(500))

E = [-0.09463371  0.01449544 -0.00146736]
Phi = 9.491845753399142e-05
speed 500 eV = 13262051.164024979


In [6]:
sample_y_bounds, sample_z_bounds = sample_bounds(meshes)

y_center = 0.5 * (sample_y_bounds[0] + sample_y_bounds[1])
z_center = 0.5 * (sample_z_bounds[0] + sample_z_bounds[1])

p0 = np.array([0.75 * field["h"], y_center, z_center])
print("grid classification:", classify_grid_point(p0, field))

grid classification: {'status': 'free', 'i': 167, 'j': 166, 'k': 166, 'owner_id': 0}


In [5]:
from rfa_model.collisions import *
from rfa_model.geometry import build_collision_mesh_dict

collision_meshes_primary = build_collision_mesh_dict(
    meshes,
    frame_meshes,
    include_sample=True,
)

collision_meshes_emit = build_collision_mesh_dict(
    meshes,
    frame_meshes,
    include_sample=False,
)

collision_mesh, face_owner, intersector = build_stl_intersector(
    collision_meshes_primary
)

stl_boxes = build_stl_bounding_boxes(
    collision_meshes_primary,
    padding=1.0e-3,
)

print("combined collision mesh faces:", len(collision_mesh.faces))
print("face_owner length:", len(face_owner))
print("number of STL boxes:", len(stl_boxes))

combined collision mesh faces: 329352
face_owner length: 329352
number of STL boxes: 10


In [6]:
p0 = np.array([0.001, 0.0, 0.0])
p1 = np.array([0.08, 0.0, 0.0])

hit_grid = first_analytic_grid_hit(p0, p1, field)

print(hit_grid)

None


In [7]:
p0 = np.array([0.001, 0.01, 0.0])
p1 = np.array([0.08, 0.01, 0.0])

hit_grid = first_analytic_grid_hit(p0, p1, field)

print(hit_grid)

{'kind': 'sphere', 'location': array([0.04407011, 0.01      , 0.        ]), 'distance': np.float64(0.043070106512759415), 't': np.float64(0.5451912216804989), 'owner': 'g1_shell', 'normal': array([0.97520899, 0.22128583, 0.        ])}


In [8]:
p0 = np.array([0.001, 0.005, 0.00025])
p1 = np.array([-0.001, 0.005, 0.00025])

hit_sample = segment_hits_sample_plane(
    p0,
    p1,
    x_sample=0.0,
    sample_y_bounds=sample_y_bounds,
    sample_z_bounds=sample_z_bounds,
)

print(hit_sample)

{'kind': 'sample_plane', 'location': array([0.     , 0.005  , 0.00025]), 'distance': np.float64(0.001), 'owner': 'sample', 'normal': array([1., 0., 0.])}


In [9]:
from rfa_model.constants import *
from rfa_model.fields import *
from rfa_model.collisions import *
from rfa_model.trajectories import *
from rfa_model.geometry import build_collision_mesh_dict, sample_bounds

Ex_interp, Ey_interp, Ez_interp = build_field_interpolators(field)
Phi_interp = build_potential_interpolator(field)

collision_meshes_emit = build_collision_mesh_dict(
    meshes,
    frame_meshes,
    include_sample=False,
)

collision_mesh_emit, face_owner_emit, intersector_emit = build_stl_intersector(
    collision_meshes_emit
)

stl_boxes_emit = build_stl_bounding_boxes(
    collision_meshes_emit,
    padding=1.0e-3,
)

sample_y_bounds, sample_z_bounds = sample_bounds(meshes)

grid_transparency = {
    "g1_shell": 0.90,
    "g2_shell": 0.90,
    "g3_shell": 0.90,
}

h = field["h"]

y_center = 0.5 * (sample_y_bounds[0] + sample_y_bounds[1])
z_center = 0.5 * (sample_z_bounds[0] + sample_z_bounds[1])

p0 = np.array([0.75 * field["h"], y_center, z_center])
v0 = speed_from_energy_eV(100.0) * unit(np.array([1.0, 0.0, 0.0]))

print("p0 =", p0)
print("grid classification:", classify_grid_point(p0, field))

res = integrate_one_electron(
    p0=p0,
    v0=v0,
    field=field,
    Ex_interp=Ex_interp,
    Ey_interp=Ey_interp,
    Ez_interp=Ez_interp,
    intersector=intersector_emit,
    face_owner=face_owner_emit,
    collision_mesh=collision_mesh_emit,
    dt=1e-12,
    max_steps=20000,
    surface_eps=1e-6,
    grid_transparency=grid_transparency,
    rng=np.random.default_rng(1),
    adaptive_dt=True,
    dt_min=1e-13,
    dt_max=2e-11,
    max_step_fraction_of_h=0.40,
    stl_boxes=stl_boxes_emit,
    sample_plane_return=True,
    sample_y_bounds=sample_y_bounds,
    sample_z_bounds=sample_z_bounds,
)

print(res["reason"])
print(res["steps"])
print(res["traj"][-1])
print(res["hit_info"])

p0 = [ 0.000375 -0.00012   0.      ]
grid classification: {'status': 'free', 'i': 167, 'j': 166, 'k': 166, 'owner_id': 0}
left_grid
749
[ 8.33432957e-02 -1.20489791e-04  6.87513614e-08]
{'status': 'left_grid', 'i': 333, 'j': 166, 'k': 166, 'owner_id': None}
